# 06: Range Trees

*Authors: Felix Espey, Kevin Buchin*

This notebook serves as supplementary learning material for the course **Geometric Algorithms**.
It showcases and explains implementations of algorithms presented in the corresponding lecture, and elaborates on some practical considerations concerning their use.
Furthermore, it offers interactive visualisations and animations.

## Table of Contents

1. Introduction
2. Range Searching in One Dimension
3. Range Searching in Two Dimensions (Kd-Trees)
4. Range Trees
5. References

## 1. Introduction

This notebook focuses on the lecture regarding range searching. The first chapter goes over range searching one dimension using binary search trees and the second chapter features range searching in two dimensions using KD-Trees. We start by importing all the necessary parts from the backend.

In [1]:
from __future__ import annotations
from typing import Optional
from modules import Node, IntComparator, ComparisonResult, RangeSearchAnimator, RangeSearchMode, Point, PointSetInstance, KDTreeConstructionMode,KDTreeSearchAnimator, VisualisationTool, BinaryTreeInstance, KDTreeConstructionAnimator, Rectangle, KDTreeSearchMode
from modules.data_structures.binary_trees import EST

## 2. Range Searching in One Dimension

The goal of a one-dimensional range search is to find all entries in a set (of numbers) that are within a given range. The algorithm presented in the lecture uses a binary search tree in which the inner nodes are used solely as splitting nodes and the data is stored in the leaves.
The search starts at the root of the tree and descends until the first node within the range is found. This node is called the splitting node. Since all nodes in the left subtree are smaller than the splitting node, they are guaranteed to be smaller than the right border of the range, so a simple comparison with the left boundary of the range is enough. Similarly, all nodes in the right subtree are guaranteed to be bigger than the left border of the range, so a comparison with the right boundary is sufficient.

The algorithms include lines like:
- tag_current_node(some integer)
- go_to_left_child
- go_to_parent
All of these lines are only needed for the drawing and are not part of the actual algorithm.

The implementations below all consist of methods that take a tree node as an argument. Usually this would be implemented as methods of the node class, but since the node class is very large and features a lot of methods not needed here, it is instead moved to the backend, and the methods here take a node as an argument.

Since the drawing does not provide an easy way to change the boundaries of the range search directly, they are instead defined as variables below.

In [2]:
# -------- change boundaries here
LOWER_BOUND = 10
UPPER_BOUND = 50

# -------- visualization setup
bti = BinaryTreeInstance()
vis = VisualisationTool(400,400, bti)

# -------- compares two nodes
int_comparator = IntComparator()

### 2.1 Finding the Splitting Node

As stated before, the first step for completing a range search is to find the first node that is within the range. The method "find_splitting_node" below does exactly that, descending through the tree from the given node until a node within the range is found. An important note here is that the construction of the tree guarantees that each node has either 0 or 2 children, which simplifies some checks.
There are a total of five cases that need to be accounted for:

- The current node is a leaf and outside the range. Then no splitting node exists, and the return value is None. In the drawing, this node is marked red.
- The current node is inside the range. Then the current node is returned. In the drawing, this node is marked green.
- The current node is outside the range and is not a leaf. The algorithm then further descends the tree, going right if the current node is to the left of the range and going left if the current node is on the right. In the drawing these nodes are marked blue.

In [3]:
def find_splitting_node(node : Node[int, None], lower_bound : int, upper_bound : int, rta : RangeSearchAnimator) -> Node[int, None] | None:
    cr_left = int_comparator.compare(lower_bound, node.key)
    cr_right = int_comparator.compare(upper_bound, node.key)
    if cr_right is ComparisonResult.BEFORE:
        #range fully left of node
        if node.left is None:
            rta.tag_cur_node(3)
            return None
        else:
            rta.tag_cur_node(1)
            rta.go_to_left_child()
            return find_splitting_node(node.left, lower_bound, upper_bound, rta)
    elif cr_left is ComparisonResult.AFTER:
        #range fully right of node
        if node.right is None:
            rta.tag_cur_node(3)
            return None
        else:
            rta.tag_cur_node(1)
            rta.go_to_right_child()
            return find_splitting_node(node.right, lower_bound, upper_bound, rta)
    else:
        #node in range
        rta.tag_cur_node(2)
        rta.save_node() # needed to find node in range search
        return node

# -------- visualization
def run_find_splitting_node(est : EST[int]) -> RangeSearchAnimator:
    rta = RangeSearchAnimator(est)
    if LOWER_BOUND < UPPER_BOUND:
       find_splitting_node(est._root, LOWER_BOUND, UPPER_BOUND, rta)
    return rta

vis.register_algorithm("find splitting node", run_find_splitting_node, RangeSearchMode())
vis.display()

### 2.2 Helper Methods

Once a splitting node is found, the next step is to report all nodes within the range from its left and right subtrees. First, we define a helper method to return all leaves in a given subtree.

In [4]:
def leaves(node : Node[int, None], rta:RangeSearchAnimator):
    if node.is_leaf():
        rta.tag_cur_node(2)
    else:
        rta.tag_cur_node(1)
        rta.go_to_left_child()
        leaves(node.left, rta)
        rta.go_to_parent()
        rta.go_to_right_child()
        leaves(node.right, rta)
        rta.go_to_parent()

Using this method, we can then define two additional helper methods that return all nodes smaller/larger than a given parameter.

In less_or_equal, the current node is compared to the right boundary of the range. This creates the following cases:
- The node is left of the boundary and not a leaf. Then all the leaves in the left subtree are reported, and the algorithm descends to the right child of the node.
- The node is left of the boundary and a leaf. Then the current node is reported
- The node is right of the boundary and not a leaf. Then the algorithm descends to the left child.
- The node is right of the boundary and a leaf. Then the algorithm returns without reporting the node.

In [5]:
def less_or_equal(node : Node[int, None], upper_bound : int, rta : RangeSearchAnimator):
    cr = int_comparator.compare(upper_bound, node.key)
    if cr is ComparisonResult.MATCH or cr is ComparisonResult.AFTER:
        #less than search term
        if not node.is_leaf():
            rta.tag_cur_node(1)
            rta.go_to_left_child()
            leaves(node.left, rta)
            rta.go_to_parent()
            rta.go_to_right_child()
            less_or_equal(node.right, upper_bound, rta)
            rta.go_to_parent()
        else:
            rta.tag_cur_node(2)
    else:
        #more than search term
        if not node.is_leaf():
            rta.tag_cur_node(1)
            rta.go_to_left_child()
            less_or_equal(node.left, upper_bound, rta)
            rta.go_to_parent()
        else:
            rta.tag_cur_node(3)

The greater_or_equal method is very similar, as it does the same but flips the decision and direction of descent. The cases are as follows:

- The node is right of the boundary and not a leaf. Then all the leaves in the right subtree are reported, and the algorithm descends to the left child of the node.
- The node is right of the boundary and a leaf. Then the current node is reported
- The node is left of the boundary and not a leaf. Then the algorithm descends to the right child.
- The node is left of the boundary and a leaf. Then the algorithm returns without reporting the node.

In [6]:
def greater_or_equal(node : Node[int, None], lower_bound : int, rta : RangeSearchAnimator):
    cr = int_comparator.compare(lower_bound, node.key)
    if cr is ComparisonResult.BEFORE or cr is ComparisonResult.MATCH:
        #more than search term
        if not node.is_leaf():
            rta.tag_cur_node(1)
            rta.go_to_left_child()
            greater_or_equal(node.left, lower_bound, rta)
            rta.go_to_parent()
            rta.go_to_right_child()
            leaves(node.right, rta)
            rta.go_to_parent()
        else:
            rta.tag_cur_node(2)
    else:
        #less than search term
        if not node.is_leaf():
            rta.tag_cur_node(1)
            rta.go_to_right_child()
            greater_or_equal(node.right, lower_bound, rta)
            rta.go_to_parent()
        else:
            rta.tag_cur_node(3)

The code below creates a drawing for the less_or_equal and greater_or_equal methods. The path through the tree is in blue; the reported leaves are in green. Leaves that were visited but not reported are red. It is important to note that the less_or_equal and greater_or_equal methods do not use the splitting node, instead starting at the root.

In [7]:
def run_less_or_equal(est : EST[int]) -> RangeSearchAnimator:
    rta = RangeSearchAnimator(est)
    less_or_equal(est._root,UPPER_BOUND , rta)
    return rta

def run_greater_or_equal(est : EST[int]) -> RangeSearchAnimator:
    rta = RangeSearchAnimator(est)
    greater_or_equal(est._root,LOWER_BOUND , rta)
    return rta

vis.register_algorithm("report smaller than upper bound", run_less_or_equal, RangeSearchMode())
vis.register_algorithm("report bigger than lower bound", run_greater_or_equal, RangeSearchMode())
vis.display()

### 3.3 Range Search
Using the methods defined above, we can now create a range_search algorithm. It starts by finding the splitting node for the given range. If none exists, an empty list is returned. If the splitting node is a leaf, it is returned as the solution. If a left subtree exists, all nodes greater than or equal to the left boundary of the range are added to the result. If a right subtree exists, all nodes less than or equal to the right boundary of the range are also added.

As before, the drawing marks all visited nodes in blue, all nodes within the range in green, and all leaves that were visited but not reported in red.

In [8]:
def range_search(node : Node[int, None], lower_bound: int, upper_bound : int, rta : RangeSearchAnimator) -> list[Node[int, None]]:
        splitting_node = find_splitting_node(node, lower_bound, upper_bound, rta)
        if splitting_node is None:
            return []
        if splitting_node.is_leaf():
            return [splitting_node]
        else:
            rta.load_node()
            rta.tag_cur_node(1)
            result = []
            if splitting_node.left is not None:
                rta.go_to_left_child()
                greater_or_equal(splitting_node.left, lower_bound, rta)
                rta.go_to_parent()
            if splitting_node.right is not None:
                rta.go_to_right_child()
                less_or_equal(splitting_node.right, upper_bound, rta)
                rta.go_to_parent()
            return result

# -------- visualization
def run_range_search(est : EST[int]) -> RangeSearchAnimator:
    rta = RangeSearchAnimator(est)
    if LOWER_BOUND < UPPER_BOUND:
        range_search(est._root, LOWER_BOUND, UPPER_BOUND, rta)
    return rta

vis.register_algorithm("report in range", run_range_search, RangeSearchMode())
vis.display()

## 3. Range Searching in Two Dimensions (Kd-Trees)

A two-dimensional range query is conceptually a simple extension of a one-dimensional range query. The previously used binary tree splits the search space into two approximately equal-sized subspaces at each node. The two-dimensional equivalent is a kd-tree (or more accurately 2d-tree), which splits the search space along one of the two axes at each level, with each even layer splitting the x-space and each odd layer splitting the y-space.

We start with a simple implementation of a KDNode. Apart from the classic tree structure with a left and right child and a parent, it only needs to know the associated point and the axis it splits.

In [9]:
class KDNode:

    def __init__(self, point : Point, axis : int = 0,
                 left : Optional[KDNode]=None, right : Optional[KDNode]=None):
        self.point : Point = point
        self.axis : int = axis       # 0 for x, 1 for y
        self.left : Optional[KDNode] = left
        self.right : Optional[KDNode] = right

### 3.1 Constructing a KD-Tree
Given the KDNode, we can build a KDTree recursively given a list of numbers. As with the range search implementation before, all the lines containing the animator object are only used for the drawing.

The first two checks are to ensure no errors occur later. Afterward, the current axis is calculated from the given depth, and, if only one point remains in the list, a new node with no children is created and returned. Otherwise, the remaining points are sorted based on the current axis, and a new node with the median point is created, with the left and right children being obtained from the two recursive calls.

In [10]:
def build(points : list[Point], depth : int, animator : KDTreeConstructionAnimator) -> Optional[KDNode]:
    if not points or len(points) == 0:
        return None
    axis = depth % 2
    if len(points) == 1:
        animator.add_leaf(points[0])
        return KDNode(points[0], axis)
    if axis == 0:
        points.sort(key=lambda point : point.x)
    else:
        points.sort(key=lambda point : point.y)
    animator.set_current_points(points)
    median_index = (len(points) - 1) // 2
    animator.add_inner_node(points[median_index])
    node = KDNode(
        point=points[median_index],
        axis=axis
    )
    animator.go_to_left_child()
    node.left = build(points[:median_index+1], depth + 1, animator)
    animator.go_to_parent()
    animator.go_to_right_child()
    node.right = build(points[median_index+1:], depth + 1, animator)
    animator.go_to_parent()
    return node

The code below animates the construction of a KD-tree for a set of points. Each line represents an inner node of the tree. During the animation, all points within the current subtree are marked in red.

In [11]:
def build_kd_tree(points : set[Point]) -> KDTreeSearchAnimator:
    animator = KDTreeConstructionAnimator()
    build(list(points), 0, animator)
    animator.set_current_points([])
    return  animator

pIS = PointSetInstance()
pIS._default_number_of_random_points = 20
pIS._random_points_mode = 1
vis2 = VisualisationTool(400,400, pIS)
vis2.register_algorithm("build kd tree", build_kd_tree, KDTreeConstructionMode())
vis2.display()

### 3.2 Range Search

To implement the range search, we first need to define an outer boundary, called the global region, which contains all the points. Since the drawings start at (0,0) and goes to (400,400), no points can be outside this area, but for a more general implementation, one would have to take the min/max x/y coordinates over the point set.

Additionally, we define the search area. As stated in the previous section, there is no nice way to add this to the drawing, so we define four global variables.

In [12]:

GLOBAL_REGION = Rectangle(Point(0, 0), Point(400, 400))
global_left = GLOBAL_REGION.left
global_right = GLOBAL_REGION.right
global_lower = GLOBAL_REGION.lower
global_upper = GLOBAL_REGION.upper

LEFT = 100
RIGHT = 300
LOWER = 100
UPPER = 300
SEARCH_REGION = Rectangle(Point(LEFT, LOWER), Point(RIGHT, UPPER))

Next, we define a method that returns all points stored in the leaves of a given subtree.

In [13]:
def get_all_nodes(node : KDNode, animator : KDTreeSearchAnimator) -> list[KDNode]:
    if node.left is None and node.right is None:
        animator.add_point(node.point)
        return [node]
    ret = get_all_nodes(node.left, animator)
    ret += get_all_nodes(node.right, animator)
    return ret

The range search method takes four parameters. Three are pretty clear: the current node, the animator, and the search region. The last parameter keeps track of the region of the current subtree. This region starts out as the global search region defined earlier and is split along the current axis in each recursive call to calculate the regions of the left and right child. This region is then compared to the search region, creating three different cases:
- The areas have no overlap. The subtree is skipped entirely.
- The current region is fully inside the search region. All points in the subtree are returned.
- The two regions overlap in part. The range search is continued in the subtree.

In [14]:
def kd_range_search(node : KDNode, animator : KDTreeSearchAnimator, search_region : Rectangle, cur_region : Rectangle) -> list[KDNode]:
    if node is None:
        return []
    if node.left is None and node.right is None:
        if search_region.contains_point(node.point):
            animator.add_point(node.point)
            return [node]

    lc_halfplane = Rectangle(Point(global_left,global_lower), Point(node.point.x,global_upper)) if node.axis == 0 \
        else Rectangle(Point(global_left,global_lower), Point(global_right,node.point.y))
    rc_halfplane = Rectangle(Point(node.point.x,global_lower), Point(global_right,global_upper)) if node.axis == 0 \
        else Rectangle(Point(global_left,node.point.y), Point(global_right,global_upper))
    lc_region = cur_region.intersection(lc_halfplane)
    rc_region = cur_region.intersection(rc_halfplane)

    nodes : list[KDNode]= []

    if search_region.contains_rectangle(lc_region):
        animator.set_current_region(lc_region)
        nodes.extend(get_all_nodes(node.left, animator))
        animator.set_current_region(cur_region)
    elif search_region.intersection(lc_region) is not None:
        animator.set_current_region(lc_region)
        nodes.extend(kd_range_search(node.left, animator, search_region, lc_region))
        animator.set_current_region(cur_region)

    if search_region.contains_rectangle(rc_region):
        animator.set_current_region(rc_region)
        nodes.extend(get_all_nodes(node.right, animator))
        animator.set_current_region(cur_region)
    elif search_region.intersection(rc_region) is not None:
        animator.set_current_region(rc_region)
        nodes.extend(kd_range_search(node.right, animator, search_region, rc_region))
        animator.set_current_region(cur_region)
    return nodes

The drawing below marks all reported points in red. During the animation, the search region is drawn in semi-transparent red and the current region in non-transparent red.

In [15]:
def kd_range_search_alg(points : set[Point]):
    construction_animator = KDTreeConstructionAnimator()
    root = build(list(points), 0, construction_animator)
    search_animator = KDTreeSearchAnimator(construction_animator, SEARCH_REGION, GLOBAL_REGION)
    kd_range_search(root, search_animator, SEARCH_REGION, GLOBAL_REGION)
    search_animator.set_current_region(None)
    return search_animator

pIS._default_number_of_random_points = 5
vis2.register_algorithm("range search", kd_range_search_alg, KDTreeSearchMode())
vis2.display()